# Jiaozi — one Kaggle notebook, change only the competition name

This notebook runs the full Jiaozi flow (data → Module 3 model selection → Module 4
code generation → training → submission) on a Kaggle competition.

**To switch competitions, edit `BENCHMARK_KEY` in the config cell (Section 3) — nothing else.**
Everything else (data location, labels, metric, submission format) is read from
`vision_benchmark_catalog.py` from Jiaozi `main`.

Covered competitions (classification, fully automatic):
`plant_pathology_2020`, `aptos2019`, `dog_breed`, `leaf_classification`,
`aerial_cactus`, `dogs_vs_cats_redux`, `histopathologic_cancer`.

Also in the catalog but **not** served by this classification notebook (need a
detection / segmentation / denoising pipeline): `global_wheat`, `ultrasound_nerve`,
`denoising_dirty_documents`. The notebook detects these and stops with a clear message.

## 1. Clone Jiaozi main and install dependencies

The repository is cloned from `Isso-W/Jiaozi` at `main`, then the runtime changes into
`experiments/mlestar_kaggle_benchmarks` for the isolated experiment environment.

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

REPO_URL = 'https://github.com/Isso-W/Jiaozi.git'
REPO_ROOT = Path('/content/Jiaozi') if Path('/content').exists() else Path.cwd() / 'Jiaozi'
PROJECT_DIR = REPO_ROOT / 'experiments' / 'mlestar_kaggle_benchmarks'

if REPO_ROOT.exists():
    shutil.rmtree(REPO_ROOT)
subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'main', REPO_URL, str(REPO_ROOT)], check=True)
os.chdir(PROJECT_DIR)
for path in (PROJECT_DIR, REPO_ROOT):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
print('Commit:'); subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], check=False)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[vision,llm,kaggle]'], cwd=PROJECT_DIR, check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
    'chromadb', 'datasets', 'huggingface_hub', 'kaggle>=2.2.2', 'networkx',
    'pandas', 'pillow', 'scikit-learn', 'sentence-transformers', 'timm', 'transformers',
], check=True)
print('Ready.')

## 2. Configure secrets (Kaggle required, Hugging Face recommended)

Set exactly one Kaggle credential in Colab's left-sidebar Secrets: `KAGGLE_API_TOKEN`.
Optionally set `HF_TOKEN` for gated checkpoints such as DINOv3. Neither secret is printed or persisted.
You must also accept each competition's rules once on its Kaggle page.

In [ ]:
import os
try:
    from google.colab import userdata
except ImportError as error:
    raise RuntimeError('This credential cell must run in Google Colab.') from error

kaggle_token = userdata.get('KAGGLE_API_TOKEN')
if not kaggle_token:
    raise RuntimeError('Set KAGGLE_API_TOKEN in Colab Secrets.')
os.environ['KAGGLE_API_TOKEN'] = kaggle_token
del kaggle_token

hf_token = userdata.get('HF_TOKEN')

if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    os.environ["HUGGINGFACE_HUB_TOKEN"] = hf_token
del hf_token

## 3. THE ONLY CELL TO EDIT — pick the competition

Set `BENCHMARK_KEY` to any catalog key or raw Kaggle slug. The rest are optional knobs.

In [ ]:
# >>> change this line only <<<
BENCHMARK_KEY = 'plant_pathology_2020'

# Optional knobs (sensible defaults):
# NOTE: DINOv3 partial finetune needs ~30-40 epochs to converge; 10-15 undertrains
# and costs several points. The metric is read from the catalog (Plant Pathology = roc_auc).
EPOCHS      = 40          # training epochs
IMAGE_SIZE  = 384         # target resolution; auto-snapped per backbone, Swin pinned to 224
TTA         = ['hflip', 'vflip', 'rot90']   # test-time augmentation views
SUBMIT      = False       # set True to actually submit to Kaggle
PRIORITY    = 'accuracy'  # Module 3 priority: speed | accuracy | balanced

DATA_ROOT   = '/content/kaggle_data'
OUTPUT_DIR  = f'/content/kaggle_run/{BENCHMARK_KEY}'
os.environ.setdefault('M4_LLM_PROVIDER', 'none')  # deterministic template code-gen
print('BENCHMARK_KEY =', BENCHMARK_KEY)

## 4. Resolve the competition and route by task type

Classification competitions continue automatically; detection / segmentation /
denoising stop here because they need a different pipeline.

In [ ]:
from vision_benchmark_catalog import get_benchmark

bench = get_benchmark(BENCHMARK_KEY)
task_type = bench.get('task_type', 'classification')
print(f"name       : {bench['name']}")
print(f"competition: {bench.get('competition')}")
print(f"task_type  : {task_type}")
print(f"metric     : {bench.get('metric')}")

if task_type != 'classification':
    raise SystemExit(
        f"'{BENCHMARK_KEY}' is a {task_type} task. This classification notebook does not "
        f"serve it — it needs a {task_type}-specific data loader, model, and submission "
        f"format. Note: {bench.get('notes', '')}"
    )

## 5. Download data + generate the project (Module 3 → Module 4)

In [ ]:
import subprocess, sys

cmd = [sys.executable, 'run_kaggle_benchmark.py', BENCHMARK_KEY,
       '--data-root', DATA_ROOT, '--output', OUTPUT_DIR, '--priority', PRIORITY]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, cwd=str(REPO_ROOT), check=True)

MODULE4_DIR = Path(OUTPUT_DIR) / 'module4_code'
CONFIG_PATH = MODULE4_DIR / 'configs.json'
assert CONFIG_PATH.exists(), CONFIG_PATH
print('Generated project:', MODULE4_DIR)

## 6. Inject the improvements into the generated config

Raises resolution (with per-backbone patch-grid snapping; Swin stays 224), enables
multi-view TTA, and shrinks the batch above 384 to bound GPU memory.

In [ ]:
import json

def safe_image_size(backbone, checkpoint, requested):
    ident = f"{str(backbone).lower()} {str(checkpoint).lower()}"
    if 'swin' in ident or 'window7-224' in ident:
        return min(requested, 224)
    patch = 14 if 'dinov2' in ident else 16
    return max((requested // patch) * patch, patch)

configs = json.loads(CONFIG_PATH.read_text())
tta_cfg = {'enabled': True, 'num_augments': len(TTA) + 1, 'transforms': TTA}
for cfg in configs:
    mc = cfg.setdefault('model_config', {})
    backbone = cfg.get('backbone') or mc.get('backbone')
    ckpt = cfg.get('checkpoint') or mc.get('pretrained_hf_id') or mc.get('checkpoint')
    size = safe_image_size(backbone, ckpt, IMAGE_SIZE)
    for d in (cfg, mc):
        d['image_size'] = size
        d['tta'] = tta_cfg
        if size >= 384:
            d['batch_size'] = min(int(d.get('batch_size', 16) or 16), 8)
            d['eval_batch_size'] = min(int(d.get('eval_batch_size', 32) or 32), 16)
CONFIG_PATH.write_text(json.dumps(configs, indent=2))
for c in configs:
    print(c.get('backbone'), '| image_size', c['image_size'], '| tta', c['model_config']['tta']['transforms'])

## 7. Train

In [ ]:
import subprocess, sys, torch
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
cmd = [sys.executable, '-u', 'run.py', '--config', 'configs.json', '--epochs', str(EPOCHS)]
print('Running:', ' '.join(cmd), 'cwd=', MODULE4_DIR)
subprocess.run(cmd, cwd=str(MODULE4_DIR), check=True)

## 8. Predict the test set + write (optionally submit) the submission

`kaggle_submit.py` formats predictions to match the competition's sample submission —
probability-per-class columns, single positive-class probability (ROC AUC / log loss),
or a single label — all driven by the catalog entry.

In [ ]:
import subprocess, sys
receipt = Path(OUTPUT_DIR) / 'submission_receipt.json'
cmd = [sys.executable, 'kaggle_submit.py', BENCHMARK_KEY,
       '--project', str(MODULE4_DIR), '--data-root', DATA_ROOT,
       '--receipt-out', str(receipt), '--log-memory']
if SUBMIT:
    cmd.append('--submit')
print('Running:', ' '.join(cmd))
subprocess.run(cmd, cwd=str(REPO_ROOT), check=True)
print(receipt.read_text())

## 9. Inspect the submission file

In [ ]:
import pandas as pd
sub = pd.read_csv(MODULE4_DIR / 'submission.csv')
print(sub.shape)
display(sub.head())